In [15]:
import tensorflow as tf
model = tf.keras.models.load_model("model/deepfake_tf219.h5")
print("Model loaded OK")


Model loaded OK


In [16]:
import numpy as np

X_train = np.load("data/X_train.npy")

train_mean = X_train.mean()
train_std  = X_train.std()

print("train_mean:", train_mean)
print("train_std:", train_std)


train_mean: -8.4896104e-11
train_std: 0.99999785


In [18]:
import librosa
import numpy as np

def preprocess_audio(path):
    y, sr = librosa.load(path, sr=16000, mono=True)

    mfcc = librosa.feature.mfcc(
        y=y,
        sr=sr,
        n_mfcc=126
    )  # (126, time)

    # fix time axis
    if mfcc.shape[1] < 154:
        mfcc = np.pad(mfcc, ((0,0),(0,154-mfcc.shape[1])))
    else:
        mfcc = mfcc[:, :154]

    mfcc = mfcc.T  # (154, 126)

    # 🔥 PER-SAMPLE NORMALIZATION (CRITICAL)
    mfcc = (mfcc - mfcc.mean()) / (mfcc.std() + 1e-8)

    mfcc = mfcc[..., np.newaxis]
    mfcc = mfcc[np.newaxis, ...]

    return mfcc


In [19]:
x1 = preprocess_audio("test_audio/saiteja.wav")
x2 = preprocess_audio("test_audio/deepfake.wav")

p1 = model.predict(x1)[0][0]
p2 = model.predict(x2)[0][0]

print("p1:", p1)
print("p2:", p2)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 209ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
p1: 1.8074447e-17
p2: 1.4348369e-27


In [20]:
# run this where model and preprocess_audio are available
import numpy as np

x1 = preprocess_audio("test_audio/saiteja.wav")
x2 = preprocess_audio("test_audio/deepfake.wav")

print("x1 shape, mean, std:", x1.shape, float(x1.mean()), float(x1.std()))
print("x2 shape, mean, std:", x2.shape, float(x2.mean()), float(x2.std()))


x1 shape, mean, std: (1, 154, 126, 1) -1.5727467106430026e-09 1.0
x2 shape, mean, std: (1, 154, 126, 1) 1.965933416059329e-09 1.0


In [21]:
import numpy as np
print("X_train dtype:", np.load("data/X_train.npy").dtype)

# enforce dtype before predicting
x1 = x1.astype(np.float32)
x2 = x2.astype(np.float32)


X_train dtype: float32


In [24]:
import numpy as np

X_dev = np.load("data/X_dev.npy")
y_dev = np.load("data/y_dev.npy")

# add channel dimension
X_dev = X_dev[..., np.newaxis]

# pick FAKE samples
idx = np.where(y_dev == 1)[0][:5]

print("Selected indices:", idx)
print("True labels:", y_dev[idx])

preds = model.predict(X_dev[idx]).flatten()
print("Predictions:", preds)


Selected indices: [17 19 30 36 46]
True labels: [1 1 1 1 1]
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 509ms/step
Predictions: [0.7336181 0.8833213 0.9034666 0.8075839 0.9514915]
